In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Three-Qubit Gates
#Amitava Chakraborty - 15th Jan 2018
from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

In [ ]:
# Useful additional packages
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from math import pi
# Importing standard Qiskit libraries and configuring account
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_backend_config import execute
from qiskit.visualization import circuit_drawer
from qiskit.quantum_info import state_fidelity
from qiskit_backend_config import BasicAer

backend = BasicAer.get_backend('unitary_simulator')

In [ ]:
#Non unitary operations
#Now we have gone through all the unitary operations in quantum circuits 
#we also have access to non-unitary operations. 
#These include measurements, reset of qubits, and classical conditional operations.
q = QuantumRegister(1)
c = ClassicalRegister(1)

#Measurements
#We don’t have access to all the information when we make a measurement in a quantum computer. 
#The quantum state is projected onto the standard basis. 
#Below are two examples showing a circuit that is prepared in a basis state and 
#the quantum computer prepared in a superposition state.
qc = QuantumCircuit(q, c)
qc.measure(q, c)
qc.draw(output='mpl')

In [ ]:
backend = BasicAer.get_backend('qasm_simulator')
job = execute(qc, backend, shots=1024)
job.result().get_counts(qc)
#The simulator predicts that 100 percent of the time the classical register returns 0.

In [ ]:
qc = QuantumCircuit(q, c)
qc.h(q)
qc.measure(q, c)
qc.draw(output='mpl')

In [ ]:
job = execute(qc, backend, shots=1024)
job.result().get_counts(qc)
#The simulator predicts that 50 percent of the time the classical register returns 0 or 1.

In [ ]:
#Reset
#It is also possible to reset qubits to the |0> state in the middle of computation. 
#Note that reset is not a Gate operation, since it is irreversible.
qc = QuantumCircuit(q, c)
qc.reset(q[0])
qc.measure(q, c)
qc.draw(output='mpl')

In [ ]:
job = execute(qc, backend, shots=1024)
job.result().get_counts(qc)

In [ ]:
qc = QuantumCircuit(q, c)
qc.h(q)
qc.reset(q[0])
qc.measure(q, c)
qc.draw(output='mpl')

In [ ]:
job = execute(qc, backend, shots=1024)
job.result().get_counts(qc)
#Here we see that for both of these circuits 
#the simulator always predicts that the output is 100 percent in the 0 state.

In [ ]:
#Conditional operations
#It is also possible to do operations conditioned on the state of the classical register
qc = QuantumCircuit(q, c)
with qc.if_test((c, 1)):
    qc.x(q[0])
qc.measure(q,c)
qc.draw(output='mpl')
#Here the classical bit always takes the value 0 so the qubit state is always flipped.

In [ ]:
job = execute(qc, backend, shots=1024)
job.result().get_counts(qc)

In [ ]:
qc = QuantumCircuit(q, c)
qc.h(q)
qc.measure(q,c)
with qc.if_test((c, 0)):
    qc.x(q[0])
qc.measure(q,c)
qc.draw(output='mpl')

In [ ]:
job = execute(qc, backend, shots=1024)
job.result().get_counts(qc)
#the classical bit by the first measurement is random but 
#the conditional operation results in the qubit being deterministically put into

In [ ]:
#Arbitrary initialization
#Now we want to initialize a qubit register to an arbitrary state
#An arbitrary state for n qubits may be specified by a vector of 2^n amplitudes, 
#where the sum of amplitude-norms-squared equals 1. 
#For example, the following three-qubit state can be prepared:
# Initializing a three-qubit quantum state
import math
desired_vector = [
    1 / math.sqrt(16) * complex(0, 1),
    1 / math.sqrt(8) * complex(1, 0),
    1 / math.sqrt(16) * complex(1, 1),
    0,
    0,
    1 / math.sqrt(8) * complex(1, 2),
    1 / math.sqrt(16) * complex(1, 0),
    0]

q = QuantumRegister(3)
qc = QuantumCircuit(q)

qc.initialize(desired_vector, [q[0],q[1],q[2]])

In [ ]:
backend = BasicAer.get_backend('statevector_simulator')
job = execute(qc, backend)
qc_state = job.result().get_statevector(qc)
qc_state

In [ ]:
#Fidelity is useful to check whether two states are same or not. 
#The fidelity is equal to 1 if and only if two states are same.
state_fidelity(desired_vector,qc_state)